# 📑Tutorial-9: CPA Attack on the DPA V2 Dataset

**📊 DPA V2 Dataset**

The DPA Contest V2 dataset is a side-channel analysis benchmark dataset released by the VLSI research group at Télécom ParisTech, France. As the second edition of the DPA Contest series, it focuses on evaluating differential power analysis (DPA) attack methods against hardware implementations of AES. The dataset is generated from a hardware AES module without any countermeasures, providing an "idealized" leakage-model validation environment for attack research; at the same time, by introducing high noise interference and complex signal characteristics, it simulates real-world chip operating scenarios.

In [1]:
import nuscar
import numpy as np

In [2]:
ths = nuscar.traceset.ReaderZARR('../datasets/dpa_v2.zarr')
ctn = nuscar.traceset.ContainerZARR(ths)

In [ ]:
sf = nuscar.ciphers.aes.attack_last_round_xor_hw() # Since the algorithm is a hardware implementation of AES, the AES round-10 Hamming distance model is chosen

dist = nuscar.distinguisher.CPADistinguisher() # CPA distinguisher

task = nuscar.task.DistinguisherTask(ctn, sf, distinguisher = dist, steps=1000) # Compute a result every 1000 traces, used to evaluate the minimum number of traces needed to recover the key

In [4]:
task.run()

  0%|          | 0/15000 [00:00<?, ?it/s]

2026-01-11 20:21:35,778 - nuscar.task - INFO - check selection func pass


MemoryError: Out of memory, requires [3173.78MB], system remaining [2766.84MB]

In [ ]:
# View the guessed key rank
task.show_candidate(top=4)

The following code demonstrates the process by which show_candidate recovers the key

In [ ]:
rk_recover = np.zeros(16, dtype=np.uint8)
for i in range(16):
    score = np.max(np.abs(task.result[:, i, :]), axis=1)
    rk_recover[i] = np.argmax(score)
print(rk_recover)

In [ ]:
mk = nuscar.ciphers.aes.inv_key_schedule(rk_recover)
print(mk) # Compute the master key

In [ ]:
task.show_result()

In [ ]:
# View the minimum number of traces required to successfully attack each key byte
task.show_step_result()